In [ ]:
!pip install opencv-python numpy scikit-learn ultralytics lapx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.1 MB/s eta 0:00:00


# Code


In [ ]:
import cv2
import time
import numpy as np
from collections import deque, defaultdict
from sklearn.cluster import DBSCAN
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    # Model & Classes
    YOLO_MODEL = "yolov8l.pt"
    PERSON_CLASS = 0
    BAG_CLASSES = [24, 26, 28]  # backpack, handbag, suitcase

    # Thresholds
    GROUP_EPS = 150                 # DBSCAN radius for grouping (pixels)
    ASSOCIATION_DIST_THRESH = 150   # Max dist to associate bag to person
    NEAR_GROUP_RADIUS = 150         # Dist to consider a group "attending" a bag
    ABANDON_TIME_THRESH = 5         # Secs a bag must be left alone to trigger alert
    STATIONARY_SPEED_THRESH = 3     # Max pixels/frame to be considered "stationary"
    STATIONARY_TIME_THRESH = 3      # Secs to consider person/bag "stationary"
    HISTORY_FRAMES = 30             # Number of frames to calculate speed (e.g., 1 sec at 30fps)

# ==========================================
# 2. GROUP CLUSTERER
# ==========================================
class SpatialClusterer:
    def __init__(self, eps=Config.GROUP_EPS, min_samples=1):
        self.dbscan = DBSCAN(eps=eps, min_samples=min_samples)

    def cluster(self, items_dict):
        """Clusters items based on their center (x, y) coordinates."""
        if len(items_dict) < 1:
            return {}

        coords = []
        ids = []

        for item_id, data in items_dict.items():
            coords.append([data['x'], data['y']])
            ids.append(item_id)

        labels = self.dbscan.fit_predict(np.array(coords))

        groups = defaultdict(list)
        for i, label in enumerate(labels):
            if label != -1:  # -1 represents noise in DBSCAN
                groups[label].append(ids[i])

        return groups

# ==========================================
# 3. VISUALIZER
# ==========================================
class Visualizer:
    @staticmethod
    def draw_dashed_rectangle(img, pt1, pt2, color, thickness=2, dash_length=15):
        x1, y1 = pt1
        x2, y2 = pt2

        # Draw edges with dashes
        for i in range(x1, x2, dash_length * 2):
            cv2.line(img, (i, y1), (min(i + dash_length, x2), y1), color, thickness)
            cv2.line(img, (i, y2), (min(i + dash_length, x2), y2), color, thickness)
        for i in range(y1, y2, dash_length * 2):
            cv2.line(img, (x1, i), (x1, min(i + dash_length, y2)), color, thickness)
            cv2.line(img, (x2, i), (x2, min(i + dash_length, y2)), color, thickness)

    @staticmethod
    def draw_alert(frame, x, y):
        text = "ABANDONED BAG!"
        # Add background for text readability
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 3)
        cv2.rectangle(frame, (x, y - th - 10), (x + tw, y + 10), (0, 0, 0), -1)
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.circle(frame, (x, y), 30, (0, 0, 255), 4)

    @staticmethod
    def draw_dashed_circle(img, center, radius, color, thickness=2, dash_length=15):
        import math
        # Calculate how many dashes will fit around the circumference
        circumference = 2 * math.pi * radius
        dashes = int(circumference / dash_length)

        # Draw alternating arcs
        for i in range(dashes):
            if i % 2 == 0:  # Draw every other segment to create the dash effect
                start_angle = i * (360 / dashes)
                end_angle = (i + 1) * (360 / dashes)
                cv2.ellipse(img, center, (radius, radius), 0, start_angle, end_angle, color, thickness)
# ==========================================
# 4. CORE SYSTEM PIPELINE
# ==========================================
class BaggageDetectionSystem:
    def __init__(self, config=Config):
        self.config = config
        print("Loading YOLO Model...")
        self.model = YOLO(config.YOLO_MODEL)
        self.clusterer = SpatialClusterer(eps=config.GROUP_EPS)

        # State Tracking
        self.history = {'persons': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES)),
                        'bags': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES))}
        self.stationary_timers = {'persons': {}, 'bags': {}}

        # Association Tracking
        self.bag_owner = {}         # bag_id -> person_id
        self.bag_owner_group = {}   # bag_id -> group_id
        self.bag_left_timer = {}    # bag_id -> timestamp (when left alone)

    def _compute_speed(self, history_deque):
        if len(history_deque) < 5:
            return 0
        vx = history_deque[-1][0] - history_deque[0][0]
        vy = history_deque[-1][1] - history_deque[0][1]
        return np.sqrt(vx**2 + vy**2)

    def _dist(self, p1, p2):
        return np.linalg.norm(np.array([p1['x'], p1['y']]) - np.array([p2['x'], p2['y']]))

    def process_video(self, input_path, output_path):
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {input_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        print("Pipeline running... Press Ctrl+C to stop in terminal.")

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            current_time = time.time()

            # 1. Tracking
            results = self.model.track(frame, persist=True, tracker="botsort.yaml",
                                       classes=[self.config.PERSON_CLASS] + self.config.BAG_CLASSES,
                                       verbose=False)

            tracked_persons = {}
            tracked_bags = {}

            if results[0].boxes is not None and results[0].boxes.id is not None:
                boxes_xywh = results[0].boxes.xywh.cpu().numpy()  # Centers
                boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()  # Edges
                track_ids = results[0].boxes.id.int().cpu().tolist()
                classes = results[0].boxes.cls.int().cpu().tolist()

                for xywh, xyxy, tid, cls in zip(boxes_xywh, boxes_xyxy, track_ids, classes):
                    x, y, _, _ = xywh
                    x1, y1, x2, y2 = map(int, xyxy)

                    data = {'x': x, 'y': y, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}

                    if cls == self.config.PERSON_CLASS:
                        self.history['persons'][tid].append((x, y))
                        if tid not in self.stationary_timers['persons']:
                            self.stationary_timers['persons'][tid] = current_time

                        # Reset stationary timer if moving
                        if self._compute_speed(self.history['persons'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['persons'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['persons'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_persons[tid] = data

                    elif cls in self.config.BAG_CLASSES:
                        self.history['bags'][tid].append((x, y))
                        if tid not in self.stationary_timers['bags']:
                            self.stationary_timers['bags'][tid] = current_time

                        if self._compute_speed(self.history['bags'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['bags'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['bags'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_bags[tid] = data

            # 2. Grouping (Using ALL tracked persons, not just stationary ones)
            groups = self.clusterer.cluster(tracked_persons)
            person_to_group = {pid: gid for gid, members in groups.items() for pid in members}

            # 3. Association & Behavior Check
            for bid, bag in tracked_bags.items():
                # Associate bag to nearest person if not already owned
                if bid not in self.bag_owner:
                    best_match, min_dist = None, float('inf')
                    for pid, person in tracked_persons.items():
                        dist = self._dist(person, bag)
                        if dist < self.config.ASSOCIATION_DIST_THRESH and dist < min_dist:
                            min_dist = dist
                            best_match = pid
                    if best_match:
                        self.bag_owner[bid] = best_match
                        self.bag_owner_group[bid] = person_to_group.get(best_match, None)

                # Check abandonment
                owner_id = self.bag_owner.get(bid)
                owner_group = self.bag_owner_group.get(bid)
                is_attended = False

                if owner_group is not None and owner_group in groups:
                    # Check if ANY member of the owner's group is near the bag
                    for group_member_id in groups[owner_group]:
                        if group_member_id in tracked_persons:
                            if self._dist(tracked_persons[group_member_id], bag) < self.config.NEAR_GROUP_RADIUS:
                                is_attended = True
                                break
                elif owner_id in tracked_persons:
                    # Solo owner check
                    if self._dist(tracked_persons[owner_id], bag) < self.config.NEAR_GROUP_RADIUS:
                        is_attended = True

                if not is_attended:
                    if bid not in self.bag_left_timer:
                        self.bag_left_timer[bid] = current_time
                else:
                    self.bag_left_timer.pop(bid, None)  # Reset if someone comes back

            # 4. Rendering
            annotated = frame.copy()

            # Draw standard boxes
            for pid, p in tracked_persons.items():
                color = (0, 255, 0) if p.get('stationary') else (200, 200, 200) # Green if stationary, Gray if moving
                cv2.rectangle(annotated, (p['x1'], p['y1']), (p['x2'], p['y2']), color, 2)

            for bid, b in tracked_bags.items():
                cv2.rectangle(annotated, (b['x1'], b['y1']), (b['x2'], b['y2']), (255, 0, 0), 2) # Blue for bags

            # Draw Group Rectangles using actual bounding box extremes
            for gid, members in groups.items():
                active_members = [pid for pid in members if pid in tracked_persons]
                if len(active_members) > 1: # Only draw if 2+ people in group
                    g_x1 = min(tracked_persons[pid]['x1'] for pid in active_members) - 20
                    g_y1 = min(tracked_persons[pid]['y1'] for pid in active_members) - 20
                    g_x2 = max(tracked_persons[pid]['x2'] for pid in active_members) + 20
                    g_y2 = max(tracked_persons[pid]['y2'] for pid in active_members) + 20

                    Visualizer.draw_dashed_rectangle(annotated, (g_x1, g_y1), (g_x2, g_y2), (0, 255, 255), 2)
                    cv2.putText(annotated, f"Group {gid}", (g_x1, g_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Draw Alerts
            # Draw Alerts & "Safe Zone" Boundaries
            for bid, bag in tracked_bags.items():
                if bag.get('stationary', False) and bid in self.bag_left_timer:
                    # The person has stepped out of the safe zone, timer is ticking

                    bag_center = (int(bag['x']), int(bag['y']))
                    time_left_alone = current_time - self.bag_left_timer[bid]

                    if time_left_alone >= self.config.ABANDON_TIME_THRESH:
                        # 🚨 Trigger Abandoned Bag Alert!
                        Visualizer.draw_alert(annotated, bag_center[0], bag_center[1])

                        # Draw the Pink Dotted Circle representing the Safe Zone
                        # Pink in BGR is (255, 0, 255)
                        Visualizer.draw_dashed_circle(
                            annotated,
                            center=bag_center,
                            radius=self.config.NEAR_GROUP_RADIUS,
                            color=(255, 0, 255),
                            thickness=2
                        )

            out.write(annotated)

        cap.release()
        out.release()
        print(f"Done! Saved to {output_path}")

# ==========================================
# 5. EXECUTION ENTRY POINT
# ==========================================
if __name__ == "__main__":
    system = BaggageDetectionSystem()
    system.process_video("sample2.mp4", "output_final.mp4")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLO Model...
Pipeline running... Press Ctrl+C to stop in terminal.
Done! Saved to output_final.mp4


# EXP-1

In [ ]:
#-----------------------------------
#EXP2-------------------------------
#-----------------------------------
import cv2
import time
import numpy as np
from collections import deque, defaultdict
from sklearn.cluster import DBSCAN
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    YOLO_MODEL = "yolov8l.pt"
    PERSON_CLASS = 0
    BAG_CLASSES = [24, 26, 28]

    # Thresholds
    GROUP_EPS = 250
    ASSOCIATION_DIST_THRESH = 250
    NEAR_GROUP_RADIUS = 300
    ABANDON_TIME_THRESH = 6
    STATIONARY_SPEED_THRESH = 3
    STATIONARY_TIME_THRESH = 3
    HISTORY_FRAMES = 30

    # --- NEW CONFIGS FOR OWNERSHIP ---
    MAX_BAGS_PER_PERSON = 3         # Max bags one person can own
    ASSOCIATION_TIME_THRESH = 3     # Secs a person must be near a bag to claim it


# ==========================================
# 2. GROUP CLUSTERER
# ==========================================
class SpatialClusterer:
    def __init__(self, eps=Config.GROUP_EPS, min_samples=1):
        self.dbscan = DBSCAN(eps=eps, min_samples=min_samples)

    def cluster(self, items_dict):
        """Clusters items based on their center (x, y) coordinates."""
        if len(items_dict) < 1:
            return {}

        coords = []
        ids = []

        for item_id, data in items_dict.items():
            coords.append([data['x'], data['y']])
            ids.append(item_id)

        labels = self.dbscan.fit_predict(np.array(coords))

        groups = defaultdict(list)
        for i, label in enumerate(labels):
            if label != -1:  # -1 represents noise in DBSCAN
                groups[label].append(ids[i])

        return groups

# ==========================================
# 3. VISUALIZER
# ==========================================
class Visualizer:
    @staticmethod
    def draw_dashed_rectangle(img, pt1, pt2, color, thickness=2, dash_length=15):
        x1, y1 = pt1
        x2, y2 = pt2

        # Draw edges with dashes
        for i in range(x1, x2, dash_length * 2):
            cv2.line(img, (i, y1), (min(i + dash_length, x2), y1), color, thickness)
            cv2.line(img, (i, y2), (min(i + dash_length, x2), y2), color, thickness)
        for i in range(y1, y2, dash_length * 2):
            cv2.line(img, (x1, i), (x1, min(i + dash_length, y2)), color, thickness)
            cv2.line(img, (x2, i), (x2, min(i + dash_length, y2)), color, thickness)

    @staticmethod
    def draw_alert(frame, x, y):
        text = "ABANDONED BAG!"
        # Add background for text readability
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 3)
        cv2.rectangle(frame, (x, y - th - 10), (x + tw, y + 10), (0, 0, 0), -1)
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.circle(frame, (x, y), 30, (0, 0, 255), 4)

    @staticmethod
    def draw_dashed_circle(img, center, radius, color, thickness=2, dash_length=15):
        import math
        # Calculate how many dashes will fit around the circumference
        circumference = 2 * math.pi * radius
        dashes = int(circumference / dash_length)

        # Draw alternating arcs
        for i in range(dashes):
            if i % 2 == 0:  # Draw every other segment to create the dash effect
                start_angle = i * (360 / dashes)
                end_angle = (i + 1) * (360 / dashes)
                cv2.ellipse(img, center, (radius, radius), 0, start_angle, end_angle, color, thickness)

# ==========================================
# 4. CORE SYSTEM PIPELINE
# ==========================================
class BaggageDetectionSystem:
    def __init__(self, config=Config):
        self.config = config
        print("Loading YOLO Model...")
        self.model = YOLO(config.YOLO_MODEL)
        self.clusterer = SpatialClusterer(eps=config.GROUP_EPS)

        self.history = {'persons': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES)),
                        'bags': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES))}
        self.stationary_timers = {'persons': {}, 'bags': {}}

        # Association Tracking
        self.bag_owner = {}
        self.bag_owner_group = {}
        self.bag_unattended_timer = {}

        # --- NEW TRACKERS ---
        self.bag_id_map = {}               # Maps fake new bag IDs back to their original IDs
        self.bag_potential_owners = {}     # Tracks how long people have been near unowned bags

    def _compute_speed(self, history_deque):
        if len(history_deque) < 5:
            return 0
        vx = history_deque[-1][0] - history_deque[0][0]
        vy = history_deque[-1][1] - history_deque[0][1]
        return np.sqrt(vx**2 + vy**2)

    def _dist(self, p1, p2):
        return np.linalg.norm(np.array([p1['x'], p1['y']]) - np.array([p2['x'], p2['y']]))

    def process_video(self, input_path, output_path):
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {input_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        print("Pipeline running... Press Ctrl+C to stop in terminal.")

        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            #current_time = time.time()
            frame_count += 1  # <--- ADD THIS HERE
            current_time = frame_count / fps

            # 1. Tracking
            results = self.model.track(frame, persist=True, tracker="botsort.yaml", conf=0.15,
                                       classes=[self.config.PERSON_CLASS] + self.config.BAG_CLASSES,
                                       verbose=False) #change botsort.yaml/bytetrack.yaml

            tracked_persons = {}
            tracked_bags = {}

            if results[0].boxes is not None and results[0].boxes.id is not None:
                boxes_xywh = results[0].boxes.xywh.cpu().numpy()  # Centers
                boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()  # Edges
                track_ids = results[0].boxes.id.int().cpu().tolist()
                classes = results[0].boxes.cls.int().cpu().tolist()

                for xywh, xyxy, tid, cls in zip(boxes_xywh, boxes_xyxy, track_ids, classes):
                    x, y, _, _ = xywh
                    x1, y1, x2, y2 = map(int, xyxy)

                    data = {'x': x, 'y': y, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}

                    if cls == self.config.PERSON_CLASS:
                        self.history['persons'][tid].append((x, y))
                        if tid not in self.stationary_timers['persons']:
                            self.stationary_timers['persons'][tid] = current_time

                        # Reset stationary timer if moving
                        if self._compute_speed(self.history['persons'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['persons'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['persons'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_persons[tid] = data

                    elif cls in self.config.BAG_CLASSES:
                        self.history['bags'][tid].append((x, y))
                        if tid not in self.stationary_timers['bags']:
                            self.stationary_timers['bags'][tid] = current_time

                        if self._compute_speed(self.history['bags'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['bags'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['bags'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_bags[tid] = data

            # 2. Grouping (Using ALL tracked persons, not just stationary ones)
            groups = self.clusterer.cluster(tracked_persons)
            person_to_group = {pid: gid for gid, members in groups.items() for pid in members}

            # ==========================================
            # 3. Association & Behavior Check (Strict Ownership Lock)
            # ==========================================
            for bid, bag in tracked_bags.items():
                is_attended = False

                # CONDITION 1 & 2: IF BAG HAS NO OWNER, LOOK FOR ONE
                if bid not in self.bag_owner:
                    best_match, min_dist = None, float('inf')
                    for pid, person in tracked_persons.items():
                        dist = self._dist(person, bag)
                        if dist < self.config.ASSOCIATION_DIST_THRESH and dist < min_dist:
                            min_dist = dist
                            best_match = pid

                    if best_match is not None:
                        # Someone entered the radius! Lock them as the permanent owner.
                        self.bag_owner[bid] = best_match
                        self.bag_owner_group[bid] = person_to_group.get(best_match, None)
                        is_attended = True # The bag is now attended

                # CONDITION 3 & 4: BAG ALREADY HAS A LOCKED OWNER
                else:
                    owner_id = self.bag_owner[bid]
                    owner_group = self.bag_owner_group[bid]

                    # Check if the GROUP is nearby
                    if owner_group is not None and owner_group in groups:
                        for group_member_id in groups[owner_group]:
                            if group_member_id in tracked_persons:
                                if self._dist(tracked_persons[group_member_id], bag) < self.config.NEAR_GROUP_RADIUS:
                                    is_attended = True
                                    break # Bag is safe, stop checking

                    # Or check if the SOLO OWNER is nearby
                    else:
                        if owner_id in tracked_persons:
                            if self._dist(tracked_persons[owner_id], bag) < self.config.NEAR_GROUP_RADIUS:
                                is_attended = True

                # ==========================================
                # TIMER LOGIC
                # ==========================================
                if not is_attended:
                    # Case A: Born with no owner and no one is around.
                    # Case B: Locked owner/group walked away.
                    # Strangers walking by will NOT make is_attended = True.
                    if bid not in self.bag_unattended_timer:
                        self.bag_unattended_timer[bid] = current_time
                else:
                    # The bag is attended by its rightful owner/group. Reset the timer!
                    self.bag_unattended_timer.pop(bid, None)


            # ==========================================
            # 4. Rendering
            # ==========================================
            annotated = frame.copy()

            # Create a mapping of owner_id -> list of bag_ids they own
            owner_to_bags = defaultdict(list)
            for bag_id, owner_id in self.bag_owner.items():
                owner_to_bags[owner_id].append(bag_id)

            # Draw standard boxes for persons
            for pid, p in tracked_persons.items():

                # Check if this person is a locked owner
                if pid in owner_to_bags:
                    # Get all bags owned by this person and format them into a string
                    owned_bags = owner_to_bags[pid]
                    bags_str = ",".join(map(str, owned_bags))

                    color = (0, 165, 255)  # Orange in BGR
                    label = f"ID: {pid} (Bag: {bags_str})"

                    # Draw a solid orange triangle pointing down at their head
                    pt1 = (int(p['x']) - 10, p['y1'] - 35)
                    pt2 = (int(p['x']) + 10, p['y1'] - 35)
                    pt3 = (int(p['x']), p['y1'] - 20)
                    triangle_cnt = np.array([pt1, pt2, pt3])
                    cv2.drawContours(annotated, [triangle_cnt], 0, color, -1)

                else:
                    # Normal coloring: Green if stationary, Gray if moving
                    color = (0, 255, 0) if p.get('stationary') else (200, 200, 200)
                    label = f"ID: {pid}"

                # Draw the bounding box and the ID/Label
                cv2.rectangle(annotated, (p['x1'], p['y1']), (p['x2'], p['y2']), color, 2)
                cv2.putText(annotated, label, (p['x1'], p['y1'] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Draw standard boxes for bags
            for bid, b in tracked_bags.items():
                cv2.rectangle(annotated, (b['x1'], b['y1']), (b['x2'], b['y2']), (255, 0, 0), 2) # Blue for bags

                # Add the Bag ID to the blue box so you can match it to the owner!
                cv2.putText(annotated, f"Bag: {bid}", (b['x1'], b['y1'] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

            # Draw Group Rectangles using actual bounding box extremes
            for gid, members in groups.items():
                active_members = [pid for pid in members if pid in tracked_persons]
                if len(active_members) > 1: # Only draw if 2+ people in group
                    g_x1 = min(tracked_persons[pid]['x1'] for pid in active_members) - 20
                    g_y1 = min(tracked_persons[pid]['y1'] for pid in active_members) - 20
                    g_x2 = max(tracked_persons[pid]['x2'] for pid in active_members) + 20
                    g_y2 = max(tracked_persons[pid]['y2'] for pid in active_members) + 20

                    Visualizer.draw_dashed_rectangle(annotated, (g_x1, g_y1), (g_x2, g_y2), (0, 255, 255), 2)
                    cv2.putText(annotated, f"Group {gid}", (g_x1, g_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Draw Alerts & "Safe Zone" Boundaries
            for bid, bag in tracked_bags.items():
                if bag.get('stationary', False) and bid in self.bag_unattended_timer:
                    # The bag is unattended (no owner near, or born an orphan), timer is ticking

                    bag_center = (int(bag['x']), int(bag['y']))
                    time_left_alone = current_time - self.bag_unattended_timer[bid]

                    if time_left_alone >= self.config.ABANDON_TIME_THRESH:
                        # 🚨 Trigger Abandoned Bag Alert!
                        Visualizer.draw_alert(annotated, bag_center[0], bag_center[1])

                        # Draw the Pink Dotted Circle representing the Safe Zone
                        Visualizer.draw_dashed_circle(
                            annotated,
                            center=bag_center,
                            radius=self.config.NEAR_GROUP_RADIUS,
                            color=(255, 0, 255),
                            thickness=2
                        )



            out.write(annotated)

        cap.release()
        out.release()
        print(f"Done! Saved to {output_path}")

# ==========================================
# 5. EXECUTION ENTRY POINT
# ==========================================
if __name__ == "__main__":
    system = BaggageDetectionSystem()
    system.process_video("sample2.mp4", "output_14-05-26.mp4")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading YOLO Model...
Pipeline running... Press Ctrl+C to stop in terminal.
Done! Saved to output_14-05-26.mp4


# Final-updated Code:

In [ ]:
pip install -U ultralytics

In [ ]:
import cv2
import time
import numpy as np
from collections import deque, defaultdict
from sklearn.cluster import DBSCAN
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    # Model & Classes
    YOLO_MODEL = "yolov8l.pt"
    PERSON_CLASS = 0
    BAG_CLASSES = [24, 26, 28]  # backpack, handbag, suitcase

    # Thresholds
    GROUP_EPS = 150                 # DBSCAN radius for grouping (pixels)
    ASSOCIATION_DIST_THRESH = 200   # Max dist to associate bag to person (ownership)
    NEAR_GROUP_RADIUS = 150         # Dist to consider a group "attending" a bag
    ABANDON_TIME_THRESH = 3 #reduced time for testing else keep 5 only         # Secs a bag must be left alone to trigger alert
    STATIONARY_SPEED_THRESH = 3     # Max pixels/frame to be considered "stationary"
    STATIONARY_TIME_THRESH = 3      # Secs to consider person/bag "stationary"
    HISTORY_FRAMES = 30             # Number of frames to calculate speed (e.g., 1 sec at 30fps)

    # --- NEW CONFIGS FOR OWNERSHIP ---
    MAX_BAGS_PER_PERSON = 3         # Max bags one person can own
    ASSOCIATION_TIME_THRESH = 3     # Secs a person must be near a bag to claim it

# ==========================================
# 2. GROUP CLUSTERER
# ==========================================
class SpatialClusterer:
    def __init__(self, eps=Config.GROUP_EPS, min_samples=1):
        self.dbscan = DBSCAN(eps=eps, min_samples=min_samples)

    def cluster(self, items_dict):
        """Clusters items based on their center (x, y) coordinates."""
        if len(items_dict) < 1:
            return {}

        coords = []
        ids = []

        for item_id, data in items_dict.items():
            coords.append([data['x'], data['y']])
            ids.append(item_id)

        labels = self.dbscan.fit_predict(np.array(coords))

        groups = defaultdict(list)
        for i, label in enumerate(labels):
            if label != -1:  # -1 represents noise in DBSCAN
                groups[label].append(ids[i])

        return groups

# ==========================================
# 3. VISUALIZER
# ==========================================
class Visualizer:
    @staticmethod
    def draw_dashed_rectangle(img, pt1, pt2, color, thickness=2, dash_length=15):
        x1, y1 = pt1
        x2, y2 = pt2

        # Draw edges with dashes
        for i in range(x1, x2, dash_length * 2):
            cv2.line(img, (i, y1), (min(i + dash_length, x2), y1), color, thickness)
            cv2.line(img, (i, y2), (min(i + dash_length, x2), y2), color, thickness)
        for i in range(y1, y2, dash_length * 2):
            cv2.line(img, (x1, i), (x1, min(i + dash_length, y2)), color, thickness)
            cv2.line(img, (x2, i), (x2, min(i + dash_length, y2)), color, thickness)

    @staticmethod
    def draw_alert(frame, x, y):
        text = "ABANDONED BAG!"
        # Add background for text readability
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1, 3)
        cv2.rectangle(frame, (x, y - th - 10), (x + tw, y + 10), (0, 0, 0), -1)
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.circle(frame, (x, y), 30, (0, 0, 255), 4)

    @staticmethod
    def draw_dashed_circle(img, center, radius, color, thickness=2, dash_length=15):
        import math
        # Calculate how many dashes will fit around the circumference
        circumference = 2 * math.pi * radius
        dashes = int(circumference / dash_length)

        # Draw alternating arcs
        for i in range(dashes):
            if i % 2 == 0:  # Draw every other segment to create the dash effect
                start_angle = i * (360 / dashes)
                end_angle = (i + 1) * (360 / dashes)
                cv2.ellipse(img, center, (radius, radius), 0, start_angle, end_angle, color, thickness)

# ==========================================
# 4. CORE SYSTEM PIPELINE
# ==========================================
class BaggageDetectionSystem:
    def __init__(self, config=Config):
        self.config = config
        print("Loading YOLO Model...")
        self.model = YOLO(config.YOLO_MODEL)
        self.clusterer = SpatialClusterer(eps=config.GROUP_EPS)

        # State Tracking
        self.history = {'persons': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES)),
                        'bags': defaultdict(lambda: deque(maxlen=config.HISTORY_FRAMES))}
        self.stationary_timers = {'persons': {}, 'bags': {}}

        # Association Tracking
        self.bag_owner = {}         # bag_id -> person_id
        self.bag_owner_group = {}   # bag_id -> group_id
        self.bag_unattended_timer = {} # bag_id -> timestamp (when left alone or born an orphan)

        # --- NEW TRACKERS ---
        self.bag_id_map = {}               # Maps fake new bag IDs back to their original IDs
        self.bag_potential_owners = {}     # Tracks how long people have been near unowned bags

    def _compute_speed(self, history_deque):
        if len(history_deque) < 5:
            return 0
        vx = history_deque[-1][0] - history_deque[0][0]
        vy = history_deque[-1][1] - history_deque[0][1]
        return np.sqrt(vx**2 + vy**2)

    def _dist(self, p1, p2):
        return np.linalg.norm(np.array([p1['x'], p1['y']]) - np.array([p2['x'], p2['y']]))

    def process_video(self, input_path, output_path):
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            raise ValueError(f"Cannot open video file: {input_path}")

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

        print("Pipeline running... Press Ctrl+C to stop in terminal.")

        frame_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            frame_count += 1
            current_time = frame_count / fps

            # 1. Tracking
            results = self.model.track(frame, persist=True, tracker="botsort.yaml", conf=0.15,
                                       classes=[self.config.PERSON_CLASS] + self.config.BAG_CLASSES,
                                       verbose=False) #use anyone bytetrack or botsort

            tracked_persons = {}
            tracked_bags = {}

            if results[0].boxes is not None and results[0].boxes.id is not None:
                boxes_xywh = results[0].boxes.xywh.cpu().numpy()  # Centers
                boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()  # Edges
                track_ids = results[0].boxes.id.int().cpu().tolist()
                classes = results[0].boxes.cls.int().cpu().tolist()

                for xywh, xyxy, tid, cls in zip(boxes_xywh, boxes_xyxy, track_ids, classes):
                    x, y, _, _ = xywh
                    x1, y1, x2, y2 = map(int, xyxy)

                    data = {'x': x, 'y': y, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}

                    if cls == self.config.PERSON_CLASS:
                        self.history['persons'][tid].append((x, y))
                        if tid not in self.stationary_timers['persons']:
                            self.stationary_timers['persons'][tid] = current_time

                        # Reset stationary timer if moving
                        if self._compute_speed(self.history['persons'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['persons'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['persons'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_persons[tid] = data

                    elif cls in self.config.BAG_CLASSES:

                        # ==========================================
                        #  ANTI-FLICKER V2: PERSISTENT ID MAPPING
                        # ==========================================
                        # If we already mapped this new ID to an old one, swap it immediately
                        if tid in self.bag_id_map:
                            tid = self.bag_id_map[tid]
                        # If this is a brand new ID we haven't seen before...
                        elif tid not in self.history['bags']:
                            for old_tid, old_hist in self.history['bags'].items():
                                if len(old_hist) > 0:
                                    old_x, old_y = old_hist[-1]
                                    # Increased radius to 80px to catch bigger box jumps
                                    if np.sqrt((x - old_x)**2 + (y - old_y)**2) < 80:
                                        self.bag_id_map[tid] = old_tid  # Remember this mapping forever!
                                        tid = old_tid
                                        break
                        # ==========================================

                        self.history['bags'][tid].append((x, y))
                        if tid not in self.stationary_timers['bags']:
                            self.stationary_timers['bags'][tid] = current_time

                        if self._compute_speed(self.history['bags'][tid]) >= self.config.STATIONARY_SPEED_THRESH:
                            self.stationary_timers['bags'][tid] = current_time
                            data['stationary'] = False
                        else:
                            data['stationary'] = (current_time - self.stationary_timers['bags'][tid]) >= self.config.STATIONARY_TIME_THRESH

                        tracked_bags[tid] = data

            # 2. Grouping (Using ALL tracked persons, not just stationary ones)
            groups = self.clusterer.cluster(tracked_persons)
            person_to_group = {pid: gid for gid, members in groups.items() for pid in members}

            # ==========================================
            # 3. Association & Behavior Check
            # ==========================================
            # First, count how many bags each person currently owns
            owner_bag_counts = defaultdict(int)
            for owner_id in self.bag_owner.values():
                owner_bag_counts[owner_id] += 1

            for bid, bag in tracked_bags.items():
                is_attended = False

                # CONDITION 1: BAG HAS NO OWNER -> WAIT 3 SECONDS TO ASSIGN ONE
                if bid not in self.bag_owner:

                    if bid not in self.bag_potential_owners:
                        self.bag_potential_owners[bid] = {}

                    current_potentials = {}
                    best_match = None
                    min_dist = float('inf')

                    # Scan all people
                    for pid, person in tracked_persons.items():
                        dist = self._dist(person, bag)

                        # If they are inside the association radius
                        if dist < self.config.ASSOCIATION_DIST_THRESH:
                            # Carry over their existing timer, or start a new one for them
                            first_seen = self.bag_potential_owners[bid].get(pid, current_time)
                            current_potentials[pid] = first_seen

                            # Check if they have been here for >= 3 seconds AND own < 3 bags
                            if (current_time - first_seen) >= self.config.ASSOCIATION_TIME_THRESH:
                                if owner_bag_counts[pid] < self.config.MAX_BAGS_PER_PERSON:
                                    # They are eligible! Find the closest eligible person.
                                    if dist < min_dist:
                                        min_dist = dist
                                        best_match = pid

                    # Update the dictionary so people who walked away lose their timers
                    self.bag_potential_owners[bid] = current_potentials

                    # If we found an eligible owner, lock them in!
                    if best_match is not None:
                        self.bag_owner[bid] = best_match
                        self.bag_owner_group[bid] = person_to_group.get(best_match, None)
                        is_attended = True
                        # Clear potential owners memory since the bag is now claimed
                        self.bag_potential_owners.pop(bid, None)

                # CONDITION 2: BAG ALREADY HAS A LOCKED OWNER
                else:
                    owner_id = self.bag_owner[bid]
                    owner_group = self.bag_owner_group[bid]

                    # Check if the GROUP is nearby
                    if owner_group is not None and owner_group in groups:
                        for group_member_id in groups[owner_group]:
                            if group_member_id in tracked_persons:
                                if self._dist(tracked_persons[group_member_id], bag) < self.config.NEAR_GROUP_RADIUS:
                                    is_attended = True
                                    break # Bag is safe, stop checking

                    # Or check if the SOLO OWNER is nearby
                    else:
                        if owner_id in tracked_persons:
                            if self._dist(tracked_persons[owner_id], bag) < self.config.NEAR_GROUP_RADIUS:
                                is_attended = True

                # ==========================================
                # TIMER LOGIC
                # ==========================================
                if not is_attended:
                    # Case A: Born with no owner and no one is around.
                    # Case B: Locked owner/group walked away.
                    # Strangers walking by will NOT make is_attended = True.
                    if bid not in self.bag_unattended_timer:
                        self.bag_unattended_timer[bid] = current_time
                else:
                    # The bag is attended by its rightful owner/group. Reset the timer!
                    self.bag_unattended_timer.pop(bid, None)


            # ==========================================
            # 4. Rendering
            # ==========================================
            annotated = frame.copy()

            # Create a mapping of owner_id -> list of bag_ids they own
            owner_to_bags = defaultdict(list)
            for bag_id, owner_id in self.bag_owner.items():
                owner_to_bags[owner_id].append(bag_id)

            # Draw standard boxes for persons
            for pid, p in tracked_persons.items():

                # Check if this person is a locked owner
                if pid in owner_to_bags:
                    # Get all bags owned by this person and format them into a string
                    owned_bags = owner_to_bags[pid]
                    bags_str = ",".join(map(str, owned_bags))

                    color = (0, 165, 255)  # Orange in BGR
                    label = f"ID: {pid} (Bag: {bags_str})"

                    # Draw a solid orange triangle pointing down at their head
                    pt1 = (int(p['x']) - 10, p['y1'] - 35)
                    pt2 = (int(p['x']) + 10, p['y1'] - 35)
                    pt3 = (int(p['x']), p['y1'] - 20)
                    triangle_cnt = np.array([pt1, pt2, pt3])
                    cv2.drawContours(annotated, [triangle_cnt], 0, color, -1)

                else:
                    # Normal coloring: Green if stationary, Gray if moving
                    color = (0, 255, 0) if p.get('stationary') else (200, 200, 200)
                    label = f"ID: {pid}"

                # Draw the bounding box and the ID/Label
                cv2.rectangle(annotated, (p['x1'], p['y1']), (p['x2'], p['y2']), color, 2)
                cv2.putText(annotated, label, (p['x1'], p['y1'] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # Draw standard boxes for bags
            for bid, b in tracked_bags.items():
                cv2.rectangle(annotated, (b['x1'], b['y1']), (b['x2'], b['y2']), (255, 0, 0), 2) # Blue for bags

                # Add the Bag ID to the blue box so you can match it to the owner!
                cv2.putText(annotated, f"Bag: {bid}", (b['x1'], b['y1'] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

            # Draw Group Rectangles using actual bounding box extremes
            for gid, members in groups.items():
                active_members = [pid for pid in members if pid in tracked_persons]
                if len(active_members) > 1: # Only draw if 2+ people in group
                    g_x1 = min(tracked_persons[pid]['x1'] for pid in active_members) - 20
                    g_y1 = min(tracked_persons[pid]['y1'] for pid in active_members) - 20
                    g_x2 = max(tracked_persons[pid]['x2'] for pid in active_members) + 20
                    g_y2 = max(tracked_persons[pid]['y2'] for pid in active_members) + 20

                    Visualizer.draw_dashed_rectangle(annotated, (g_x1, g_y1), (g_x2, g_y2), (0, 255, 255), 2)
                    cv2.putText(annotated, f"Group {gid}", (g_x1, g_y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Draw Alerts & "Safe Zone" Boundaries
            for bid, bag in tracked_bags.items():
                if bag.get('stationary', False) and bid in self.bag_unattended_timer:
                    # The bag is unattended (no owner near, or born an orphan), timer is ticking

                    bag_center = (int(bag['x']), int(bag['y']))
                    time_left_alone = current_time - self.bag_unattended_timer[bid]

                    if time_left_alone >= self.config.ABANDON_TIME_THRESH:
                        # 🚨 Trigger Abandoned Bag Alert!
                        Visualizer.draw_alert(annotated, bag_center[0], bag_center[1])

                        # Draw the Pink Dotted Circle representing the Safe Zone
                        Visualizer.draw_dashed_circle(
                            annotated,
                            center=bag_center,
                            radius=self.config.NEAR_GROUP_RADIUS,
                            color=(255, 0, 255),
                            thickness=2
                        )

            out.write(annotated)

        cap.release()
        out.release()
        print(f"Done! Saved to {output_path}")

# ==========================================
# 5. EXECUTION ENTRY POINT
# ==========================================
if __name__ == "__main__":
    system = BaggageDetectionSystem()
    system.process_video("sample2.mp4", "output_final-15-05-26.mp4")

Loading YOLO Model...
Pipeline running... Press Ctrl+C to stop in terminal.
Done! Saved to output_final-15-05-26.mp4


# Save

In [ ]:
from google.colab import files
files.download('output_final-15-05-26.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>